<a href="https://colab.research.google.com/github/yaesur/business_python/blob/main/%EC%A7%80%EC%97%AD%EC%84%A0%ED%98%B8%EB%8F%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd

# 지역을 등급별로 구분
# '중구', '서대문구', '종로구' A등급으로 옮김
# '강동구'를 B등급으로 옮김
grade_map = {
    'S': ['강남구', '서초구', '송파구', '마포구'],
    'A': ['성동구', '용산구', '광진구', '영등포구', '동작구','중구', '서대문구', '종로구'],
    'B': ['강동구', '성북구', '양천구', '강서구', '동대문구'],
    'C': ['도봉구', '노원구', '강북구', '중랑구', '금천구', '관악구', '구로구', '은평구']
}

file = '커트라인 데이터.xlsx'
xls = pd.ExcelFile(file)
sheets = xls.sheet_names

# 순위와 점수 컬럼 설정 (매 시트마다 데이터 형식이 다르기 때문)
level_cols = [5, 5, 5, 5, 5, 5, 5, 4, 4, 7]
score_cols = [6, 6, 6, 6, 6, 6, 6, 5, 5, 8]
result = []

# 매 시트를 위 컬럼대로 불러옴 3행부터 데이터 시작이기 때문에 skiprows = 2
for i, name in enumerate(sheets):
  df = pd.read_excel(file, sheet_name=name, skiprows=2, header=None)

  data = df.iloc[:, [1, level_cols[i], score_cols[i]]]
  data.columns = ['자치구', '순위', '점수']
  result.append(data)

final = pd.concat(result).dropna()
final


,자치구,순위,점수
0,강남구,1순위,5점
1,강동구,1순위,7점
2,강동구,1순위,7점
3,강북구,1순위,5점
4,광진구,1순위,8점
...,...,...,...
175,중랑구,1순위,8점
176,중랑구,2순위,3점
177,중랑구,2순위,4점
178,중랑구,2순위,6점


In [14]:
# 순위와 점수컬럼에서 '순위'와 '점' 단위를 제거하고 수치형으로 바꿈
final['순위'] = final['순위'].astype(str).str.replace('순위', '', regex=False).str.strip()
final['순위'] = pd.to_numeric(final['순위'], errors='coerce')
final['점수'] = final['점수'].astype(str).str.replace('점', '', regex=False).str.strip()
final['점수'] = pd.to_numeric(final['점수'], errors='coerce')

# '순위'가 1인 행을 찾아서, 그 행들의 '점수' 컬럼에만 14를 더함
final.loc[final['순위'] == 1, '점수'] += 14

final['자치구'] = final['자치구'].str.strip()

# 등급과 등급 내 지역들을 1대1로 연계
def get_grade(district):
    for grade, districts in grade_map.items():
        if district in districts:
            return grade

final['등급'] = final['자치구'].apply(get_grade)

# 지역등급에 해당하는 커트라인의 점수 평균을 계산
grade_avg = final.groupby('등급')['점수'].mean().round(2).reset_index()

print(grade_avg)

  등급     점수
0  A  16.17
1  B  13.82
2  C  12.08
3  S  17.91


In [16]:
data = {'등급_숫자': [3, 2, 1, 0], '점수': [17.91, 16.17, 13.82, 12.08]} # S=3, A=2, B=1, C=0
df_corr = pd.DataFrame(data)

correlation = df_corr['등급_숫자'].corr(df_corr['점수'])

print(f"상관계수: {correlation:.4f}")
print("1에 가까울수록 등급과 점수가 정확히 비례한다는 뜻입니다.")

상관계수: 0.9981
1에 가까울수록 등급과 점수가 정확히 비례한다는 뜻입니다.
